In [2]:
import subprocess
from functools import reduce
import matplotlib.pyplot as plt
import numpy as np
from analysis import generate_data

kernel_x_y_fc = (5, 5, 5) # x y <channels> fc
compute_time = []
image_size = []
temp = []
scale = 1
for i in range(100,scale*1300,100):
    k=int(scale*13*(i/1300)+1)
    temp.append((i,i,k))
temp = reversed(temp)

for input_x_y_c in temp:
    data = generate_data(
        cpu_or_gpu='gpu',
        samples=2, single_batch_size=10, batch_counts=15,
        gpu_local_mem='true', cpu_threads=0,
        stride=2,
        kernel_arg="%d,%d,%d" % kernel_x_y_fc,
        input_arg="%d,%d,%d" % input_x_y_c,)
    data = np.mean(data['overall_time'])
    image_size.append(reduce(lambda x,a: x*a, input_x_y_c, 1)*10)
    compute_time.append(data)

plt.plot(image_size, compute_time, )

cpu_or_gpu='gpu'
Sample 1 of 2


KeyboardInterrupt: 

In [ ]:
input_x_y_c = (1000,1000,3)
compute_time = []
kernel_size = []
temp = []
scale=1
for i in range(2,scale*60,10):
    k=int(scale*30*(i//60)+1)
    temp.append((i,i,k))
temp = reversed(temp)
            
for kernel_x_y_fc in temp:  # x y <channels> fc
    print("      --> %d" % i)
    data = generate_data(
        cpu_or_gpu='gpu',
        samples=2, single_batch_size=10, batch_counts=15,
        gpu_local_mem='true', cpu_threads=0, stride=2,
        kernel_arg="%d,%d,%d" % kernel_x_y_fc,
        input_arg="%d,%d,%d" % input_x_y_c,)
    data = np.mean(data['overall_time'])
    kernel_size.append(reduce(lambda x,a: x*a, kernel_x_y_fc, 1))
    compute_time.append(data)

plt.plot(kernel_size, compute_time, )

In [3]:
def parse_device_info(exe_path):
    # Run the executable
    completed_process = subprocess.run(
        [exe_path],
        capture_output=True,
        text=True,
        check=False
    )
    # Parse lines with a colon
    data = {}
    for line in completed_process.stdout.splitlines():
        if ':' in line:
            key, val = line.split(':', 1)
            data[key.strip()] = val.strip()
    return data


exe_path = '../bin/device_info'
device_info = parse_device_info(exe_path)
device_info['Estimated max speed']= \
    int(float(device_info['Max clock frequency'][:-3])*
        float(device_info['Estimated total cores'])*
        2*1e6)
device_info

{'Platform': 'NVIDIA CUDA',
 'Device name': 'NVIDIA GeForce GTX 1050',
 'Device type': 'GPU',
 'Compute units (C)': '5',
 'Local mem per CU (L_dev)': '48 KiB',
 'Max WG size (WG_max)': '1024',
 'Max WI dims (x,y,z)': '1024 1024 64',
 'Compiler-max WG size (WG_kernel)': '256',
 'Static local mem (L_static)': '516 B',
 'Dynamic local mem (L_dyn per WG)': '512 B',
 'Max WIs per WG limited by local mem': '12159',
 'Usable WG size with all caps applied': '256',
 'Resident WGs per CU (local-mem-only)': '47',
 'Theoretical max WGs on device': '235',
 'Compute units': '5',
 'Estimated total cores': '640',
 'Max clock frequency': '1493 MHz',
 'Estimated max speed': 1911040000000}

In [6]:
input_x_y_c=(1000,1000,3)
kernel_x_y_fc = (10,10,5)
stride=2
batch_count = 15
single_batch_size = 10


out_size = ((input_x_y_c[0]-2)//stride, 
            (input_x_y_c[1]-2)//stride, 
            kernel_x_y_fc[2])
work_per_out_pixel = reduce(lambda x,a: x*a, 
                         (kernel_x_y_fc[0],kernel_x_y_fc[1]), 1)
pixel_count = (
        reduce(lambda x,a: x*a, out_size, 1)
        * single_batch_size * batch_count)
theory_flops_needed = (
    work_per_out_pixel
    * pixel_count
)
f
data_touched_per_out_byte=1+reduce(lambda x,a: x*a, kernel_x_y_fc, 1)*2

print('flops/byte:', 
      work_per_out_pixel/data_touched_per_out_byte)
theory_flops_needed/device_info['Estimated max speed']*1000

flops/byte: 0.0999000999000999


9.772205186704621

In [5]:
np.array(generate_data(
    cpu_or_gpu='gpu',
    samples=2, single_batch_size=single_batch_size, batch_counts=batch_count,
    gpu_local_mem='true', cpu_threads=0, stride=stride,
    kernel_arg="%d,%d,%d" % kernel_x_y_fc,
    input_arg="%d,%d,%d" % input_x_y_c,)['Kernel_time']).sum()/2

cpu_or_gpu='gpu'
Sample 1 of 2
Sample 2 of 2


1760.5715999999998